# KPI Monitoring v1
Valutazione della qualità dei dati sintetici generati dal workflow.
5 metriche: completezza, coerenza, diversità, validità formale, qualità semantica.

In [ ]:
import os, sys, json
os.chdir("/workspaces/hackaton-UNIBG-repo")
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
print("API Key:", "OK" if os.getenv("OPEN_ROUTER_KEY") else "MANCA")

API Key: OK


## 1. Caricamento dati della run

In [ ]:
from monitoring import load_run_data

RUN_ID = "notebook_v4"

records, schema = load_run_data(RUN_ID)
print(f"Run: {RUN_ID}")
print(f"Record caricati: {len(records)}")
print(f"Campi schema: {len(schema)}")
print(f"\nSchema:\n{json.dumps(schema, indent=2)[:800]}")

Run: notebook_v4
Record caricati: 0
Campi schema: 0

Schema:
[]


## 2. Metrica 1 — Completezza strutturale
Verifica che ogni campo definito nello schema sia presente in ogni record.

In [ ]:
from monitoring import check_structural_completeness

m1 = check_structural_completeness(records, schema)
print(f"Score: {m1['score']:.2%}")
print(f"Record: {m1['total_records']} | Campi schema: {m1['schema_fields']}")
print("\nCompletezza per campo:")
for field, ratio in m1['field_completeness'].items():
    bar = '█' * int(ratio * 20) + '░' * (20 - int(ratio * 20))
    print(f"  {field:<25s} {bar} {ratio:.0%}")

2026-05-28 23:50:50.662 | INFO     | monitoring:check_structural_completeness:66 - [Metrica 1] Completezza strutturale vs schema


Score: 0.00%


KeyError: 'total_records'

## 3. Metrica 2 — Coerenza interna
Verifica i constraints dello schema (es. totali = somma dei dettagli, net_worth = qty * net_price).

In [ ]:
from monitoring import check_internal_coherence

m2 = check_internal_coherence(records, schema)
print(f"Score: {m2['score']:.2%}")
print(f"Record coerenti: {m2['coherent_records']}/{m2['total_records']}")
if m2['errors']:
    print(f"\nErrori trovati ({len(m2['errors'])}):")
    for err in m2['errors'][:10]:
        print(f"  ❌ {err}")
else:
    print("✅ Nessun errore di coerenza")

## 4. Metrica 3 — Diversità
Misura l'unicità dei valori stringa tra i record. Score alto = record ben differenziati.

In [ ]:
from monitoring import check_diversity

m3 = check_diversity(records, schema)
print(f"Score: {m3['score']:.2%}")
print(f"Record totali: {m3['total_records']}")
print("\nUnicità per campo:")
for field, ratio in m3['uniqueness_by_field'].items():
    bar = '█' * int(ratio * 20) + '░' * (20 - int(ratio * 20))
    print(f"  {field:<25s} {bar} {ratio:.0%}")

## 5. Metrica 4 — Validità formale
Verifica che i formati dei campi siano corretti (date, IBAN, tax ID, email, telefono).

In [ ]:
from monitoring import check_field_validity

m4 = check_field_validity(records, schema)
print(f"Score: {m4['score']:.2%}")
if m4['field_validity']:
    print("\nValidità per campo:")
    for field, ratio in m4['field_validity'].items():
        bar = '█' * int(ratio * 20) + '░' * (20 - int(ratio * 20))
        print(f"  {field:<25s} {bar} {ratio:.0%}")
else:
    print("Nessun campo con pattern formale da validare")

## 6. Metrica 5 — Qualità semantica (LLM)
Valutazione qualitativa tramite LLM: realismo, varietà, coerenza semantica, usabilità.

In [ ]:
from monitoring import semantic_quality_check

m5 = semantic_quality_check(records, schema)
print(f"Overall score: {m5.get('overall_score', 0):.2%}")
print(f"Realismo:      {m5.get('realism_score', '?')}/10")
print(f"Varietà:       {m5.get('variety_score', '?')}/10")
print(f"Coerenza:      {m5.get('coherence_score', '?')}/10")
print(f"Usabilità:     {m5.get('usability_score', '?')}/10")
print(f"\nFeedback: {m5.get('feedback', '')}")

## 7. Score finale pesato

In [ ]:
from monitoring import compute_final_score
from IPython.display import display, HTML

metrics = {
    "structural_completeness": m1,
    "internal_coherence": m2,
    "diversity": m3,
    "field_validity": m4,
    "semantic_quality": m5,
}

final = compute_final_score(metrics)

grade_color = {
    "A": "#22c55e", "B": "#84cc16", "C": "#eab308",
    "D": "#f97316", "F": "#ef4444"
}

html = f"""
<style>
  .kpi-table {{ border-collapse: collapse; width: 100%; font-family: sans-serif; }}
  .kpi-table th {{ background: #1e293b; color: white; padding: 10px 14px; text-align: left; }}
  .kpi-table td {{ padding: 10px 14px; border-bottom: 1px solid #e2e8f0; }}
  .kpi-table tr:hover {{ background: #f8fafc; }}
  .score-bar {{ display: inline-block; height: 10px; border-radius: 5px; background: #e2e8f0; width: 120px; }}
  .score-fill {{ height: 10px; border-radius: 5px; }}
</style>
<h3>Report KPI — Run: {RUN_ID}</h3>
<table class='kpi-table'>
<tr><th>Metrica</th><th>Peso</th><th>Score</th><th>Dettaglio</th></tr>
<tr>
  <td>Completezza strutturale</td>
  <td>25%</td>
  <td><span style='color:{"green" if m1["score"]>0.8 else "orange"}'>{m1['score']:.1%}</span></td>
  <td>{m1.get('total_records', 0)} record, {m1.get('schema_fields', 0)} campi</td>
</tr>
<tr>
  <td>Coerenza interna</td>
  <td>30%</td>
  <td><span style='color:{"green" if m2["score"]>0.8 else "orange"}'>{m2['score']:.1%}</span></td>
  <td>{m2.get('coherent_records', 0)}/{m2.get('total_records', 0)} coerenti</td>
</tr>
<tr>
  <td>Diversità</td>
  <td>15%</td>
  <td><span style='color:{"green" if m3["score"]>0.5 else "orange"}'>{m3['score']:.1%}</span></td>
  <td>{m3.get('total_records', 0)} record</td>
</tr>
<tr>
  <td>Validità formale</td>
  <td>15%</td>
  <td><span style='color:{"green" if m4["score"]>0.8 else "orange"}'>{m4['score']:.1%}</span></td>
  <td>{len(m4.get('field_validity', {}))} campi validati</td>
</tr>
<tr>
  <td>Qualità semantica</td>
  <td>15%</td>
  <td><span style='color:{"green" if m5.get("overall_score",0)>0.6 else "orange"}'>{m5.get('overall_score', 0):.1%}</span></td>
  <td>{str(m5.get('feedback', ''))[:80]}</td>
</tr>
<tr style='font-weight:bold; background:#f1f5f9;'>
  <td>SCORE FINALE</td>
  <td>100%</td>
  <td><span style='font-size:1.2em; color:{grade_color.get(final["grade"], "#000")}'>{final['overall_score']:.1%}</span></td>
  <td><span style='font-size:1.2em; background:{grade_color.get(final["grade"], "#ccc")}; color:white; padding:2px 10px; border-radius:4px;'>{final['grade']}</span></td>
</tr>
</table>
"""
display(HTML(html))

## 8. Salvataggio report

In [ ]:
metrics["final_score"] = final

report_path = f"output/{RUN_ID}/monitoring_report.json"
os.makedirs(f"output/{RUN_ID}", exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False, default=str)

print(f"Report salvato: {report_path}")